In [6]:
import matplotlib.pyplot as plt
import ot
from graph_distances import *
import os
from utils_eval import *
import torch
import numpy as np
import time
import networkx as nx
from tqdm import tqdm
from torch_geometric.utils import to_networkx
import networkx as nx
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import pyvista as pv

import sys
sys.path.append("..")
from functional_swd import *

def calc_functional_SFTLB(graph0, graph1, alpha=.5, grid_n=10, n_projections=100, length_scale=.01):
    gdim0, gdim1, mindim, g0feat, g0_sort, g1feat, g1_sort = tlb_process(graph0, graph1, alpha=alpha)
    g0_sort_interp, _ = scaled_quantile_values(g0_sort, grid_n=grid_n)
    g1_sort_interp, _ = scaled_quantile_values(g1_sort, grid_n=grid_n)
    K = gaussian_kernel_covariance(_, length_scale=length_scale)
    direction_samples = sample_from_gaussian_with_covariance(K, n_projections)
    sw2 = sliced_functional_wasserstein_distance(g0_sort_interp, g1_sort_interp, direction_samples, _)
    return sw2
    
def build_graph_data(dataset):
    graphs = [to_networkx(data, to_undirected=True) for data in dataset]
    Ms, heights, features, labels = [], [], [], []

    for i, data in enumerate(dataset):
        A = nx.to_numpy_array(graphs[i])
        dist = nx.floyd_warshall_numpy(nx.Graph(A)).astype(np.float32)
        Ms.append(dist)

        n = A.shape[0]
        heights.append(np.ones(n, dtype=np.float32) / n)

        features.append(data.x if data.x is not None else torch.zeros((n, 1)))

        # Handle multi-label or single-label
        if hasattr(data, "y"):
            # If y is tensor with multiple elements, use first or customize here
            if isinstance(data.y, torch.Tensor) and data.y.numel() > 1:
                labels.append(data.y[0].item())
            else:
                labels.append(data.y.item())
        else:
            labels.append(i % 2)  # fallback label

    return Ms, heights, features, np.array(labels)

# === Distance matrix initialization ===
def init_matrices(N):
    mats = {}
    for name in ["gw", "energy", "tlb", "sgw_numint", "functional_sgw"]:
        mats[name] = np.zeros((N, N))
    times = {name: [] for name in mats}
    return mats, times

# === Compute pairwise distances ===
def compute_pairwise(Ms, heights, features, mats, times, alpha=0.5, 
                     name_ls=["gw", "energy", "tlb", "sgw_numint", "functional_sgw"],
                    grid_n=10, n_projections=100):
    np.random.seed(42)
    torch.manual_seed(42)
    N = len(Ms)
    for i in tqdm(range(N)):
        M1, h1, f1 = Ms[i], heights[i], features[i]
        for j in range(i + 1, N):
            M2, h2, f2 = Ms[j], heights[j], features[j]
            g0 = [torch.tensor(M1), torch.tensor(h1), f1]
            g1 = [torch.tensor(M2), torch.tensor(h2), f2]

            def time_call(fn, *args, key=None, **kwargs):
                t0 = time.time()
                val = fn(*args, **kwargs)
                times[key].append(time.time() - t0)
                return val

            if "gw" in name_ls:
                mats["gw"][i, j] = time_call(calc_FGW, g0, g1, alpha=1-alpha, key="gw")
            if "tlb" in name_ls:
                mats["tlb"][i, j] = time_call(calc_FTLB, g0, g1, alpha=alpha, key="tlb")
            if "sgw_numint" in name_ls:
                mats["sgw_numint"][i, j] = time_call(calc_SFTLB_via_numerical_integration, g0, g1, 
                                                     alpha=alpha, grid_n=grid_n, n_projections=n_projections, 
                                                     key="sgw_numint")
            if "energy" in name_ls:
                mats["energy"][i, j] = time_call(calc_EnergyLB, g0, g1, alpha=alpha, key="energy")
            if "functional_sgw" in name_ls:
                mats["functional_sgw"][i, j] = time_call(calc_functional_SFTLB, g0, g1, alpha=alpha, 
                                                         grid_n=grid_n, n_projections=n_projections, key="functional_sgw")

    # Symmetrize and sqrt distances
    for mat in mats.values():
        np.sqrt(mat.clip(0.), out=mat)
        mat += mat.T

# === kNN evaluation ===
def knn_acc(D, y, k=3, test_size=0.75, trials=10):
    accs = []
    for _ in range(trials):
        idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=test_size, stratify=y)
        D_train, D_test = D[np.ix_(idx_tr, idx_tr)], D[np.ix_(idx_te, idx_tr)]
        y_train, y_test = y[idx_tr], y[idx_te]

        clf = KNeighborsClassifier(n_neighbors=k, metric="precomputed")
        clf.fit(D_train, y_train)
        y_pred = clf.predict(D_test)
        accs.append(accuracy_score(y_test, y_pred))
    return np.mean(accs), np.std(accs)

def evaluate_all(mats, labels, times, name_ls=["gw", "energy", "tlb", "sgw_numint", "functional_sgw"],
                k=3, test_size=0.75, trials=10):
    for name in name_ls:
        mat = mats[name]
        finite_max = np.nanmax(mat[np.isfinite(mat)])
        matin = np.where(np.isfinite(mat), mat, finite_max)
        acc, std = knn_acc(matin, labels, k=k, test_size=test_size, trials=trials)
        print(f"{name:12s} | kNN acc: {acc:.3f} ± {std:.3f} | Avg time: {np.mean(times[name]):.4f}s")

def load_animal_dataset(N=73, k=50):
    """
    k = 25, 50, 75, 100
    """
    Ms = []
    labels = []
    heights = []
    features = []
    for i in range(N):
        tmp1 = np.loadtxt("/homes/numerik/piening/scratch/Neural GW/LGW/data/3d/Ds/D_k=" + str(k) + "_i=" + str(i))
        tmp2 = np.loadtxt("/homes/numerik/piening/scratch/Neural GW/LGW/data/3d/heights/height_k=" + str(k) + "_i=" + str(i))
        Ms.append(tmp1)
        heights.append(tmp2)
        features.append(torch.zeros((k, 1)))
    lengths = [11, 10, 11, 10, 10, 11, 10]
    for label_num, label_len in enumerate(lengths):
        labels += label_len * [label_num]
    return Ms, heights, features, np.array(labels)

def gen_graph_from_surface(X,Tris):
    G = nx.Graph()
    G.add_nodes_from(range(len(X)))
    for i in range(len(Tris)):
        tri = Tris[i]
        for l1 in range(3):
            for l2 in range(l1+1,3):
                G.add_edge(tri[l1],tri[l2],weight = np.linalg.norm(X[tri[l1]] - X[tri[l2]]))
    return G

def load_FAUST(pointnum=70, squared=False):
    #target_reduction = .99 #parameter for mesh decimation (higher parameter = smaller mesh)
    target_reduction = 1 - pointnum/6890 
    # target_reduction *= .9998 # caution
    f_main = "/homes/numerik/piening/scratch/Neural GW/tangential-GW-barycenter/MPI-FAUST/training/registrations/"
    #LOAD FAUST dataset
    meshes_faust = []
    labels = []
    Ms = []
    features = []
    heights = []
    print("Loading and processing FAUST")
    pbar = tqdm(range(100))
    for i in pbar:
        filepath= f_main + "tr_reg_{0}.ply".format(str("{:03d}".format(i)))
        labels.append(i//10)
        mesh = pv.read(filepath)
        
        mesh = mesh.decimate(target_reduction)
        pbar.set_description(f"Points: {len(mesh.points)}")
        mesh.rotate_x(65)
        mesh.rotate_z(170)
        mesh.rotate_y(-15)
        # meshes_faust.append(mesh)
        X=mesh.points
        Tris=mesh.faces.reshape((-1, 4))[:,1:]
        G = gen_graph_from_surface(X,Tris)
        dic = dict(nx.weighted.all_pairs_dijkstra_path_length(G))
        g = np.zeros((len(G.nodes),len(G.nodes)))
        for key in dic.keys():
            g[int(key),np.array(list(dic[key].keys()),dtype=int)] = np.array(list(dic[key].values()))
        g = (1/2) * (g + g.T)
        if squared:
            g = g**2
        Ms.append(g[:pointnum, :pointnum])
        heights.append(ot.unif(pointnum))
        features.append(torch.tensor((X[:pointnum])))
    return Ms, heights, features,  np.array(labels, dtype=float)

def load_shapes_dataset(noise=0., squared=False):
    """
    k = 25, 50, 75, 100
    """
    N = 80
    num_per_class = 20
    Ms = []
    features = []
    posns = np.load("/homes/numerik/piening/scratch/Neural GW/LGW/data/dithPos.dat",allow_pickle=True)
    n_points = 50
    heights = [np.ones(n_points) / n_points for i in range(len(posns))]
    classes = np.arange(4)
    labels = np.concatenate([np.ones(num_per_class)*i for i in classes]) 
    for i in range(N):
        points = posns[i] + noise * np.random.randn(*posns[i].shape)
        M = ot.dist(points, points, metric="euclidean")
        if squared:
            M = M**2
        Ms.append(M/np.max(M))
        features.append(torch.zeros((n_points, 1)))
        
    return Ms, heights, features, np.array(labels)

if __name__ == "__main__":
    np.random.seed(43) # changed from 42 in original code
    torch.manual_seed(43)
    name_ls = ["gw", "energy", "tlb", "sgw_numint", "functional_sgw"]
    Ms, heights, features, labels = load_shapes_dataset()
    mats, times = init_matrices(len(Ms))
    st = time.time()
    compute_pairwise(Ms, heights, features, mats, times, alpha=0.0, name_ls=name_ls, grid_n=10, n_projections=100)
    print(f"Pairwise computation finished in {time.time() - st:.2f}s")
    evaluate_all(mats, labels, times, name_ls, k=3, test_size=0.75, trials=1000)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 80/80 [00:22<00:00,  3.53it/s]


Pairwise computation finished in 22.69s
gw           | kNN acc: 0.997 ± 0.006 | Avg time: 0.0027s
energy       | kNN acc: 0.997 ± 0.009 | Avg time: 0.0007s
tlb          | kNN acc: 1.000 ± 0.002 | Avg time: 0.0007s
sgw_numint   | kNN acc: 0.995 ± 0.012 | Avg time: 0.0014s
functional_sgw | kNN acc: 0.995 ± 0.012 | Avg time: 0.0016s


In [7]:
if __name__ == "__main__":
    np.random.seed(43)
    torch.manual_seed(43)
    name_ls = ["gw", "energy", "tlb", "sgw_numint", "functional_sgw"]
    Ms, heights, features, labels = load_animal_dataset()
    mats, times = init_matrices(len(Ms))
    st = time.time()
    compute_pairwise(Ms, heights, features, mats, times, alpha=0.0, name_ls=name_ls, grid_n=10, n_projections=100)
    print(f"Pairwise computation finished in {time.time() - st:.2f}s")
    evaluate_all(mats, labels, times, name_ls, k=3, test_size=0.75, trials=1000)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 73/73 [00:23<00:00,  3.13it/s]


Pairwise computation finished in 23.34s
gw           | kNN acc: 1.000 ± 0.000 | Avg time: 0.0049s
energy       | kNN acc: 0.978 ± 0.018 | Avg time: 0.0006s
tlb          | kNN acc: 1.000 ± 0.000 | Avg time: 0.0007s
sgw_numint   | kNN acc: 0.993 ± 0.010 | Avg time: 0.0012s
functional_sgw | kNN acc: 0.991 ± 0.013 | Avg time: 0.0014s


In [8]:
if __name__ == "__main__":
    np.random.seed(43)
    torch.manual_seed(43)
    name_ls = ["gw", "energy", "tlb", "sgw_numint", "functional_sgw"]
    Ms, heights, features, labels = load_FAUST(500)
    mats, times = init_matrices(len(Ms))
    st = time.time()
    compute_pairwise(Ms, heights, features, mats, times, alpha=0.0, name_ls=name_ls, grid_n=10, n_projections=100)
    print(f"Pairwise computation finished in {time.time() - st:.2f}s")
    evaluate_all(mats, labels, times, name_ls, k=3, test_size=0.75, trials=1000)

Loading and processing FAUST


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [39:02<00:00, 23.43s/it]


Pairwise computation finished in 2342.62s
gw           | kNN acc: 0.292 ± 0.044 | Avg time: 0.4196s
energy       | kNN acc: 0.377 ± 0.056 | Avg time: 0.0080s
tlb          | kNN acc: 0.367 ± 0.056 | Avg time: 0.0224s
sgw_numint   | kNN acc: 0.376 ± 0.054 | Avg time: 0.0130s
functional_sgw | kNN acc: 0.386 ± 0.057 | Avg time: 0.0099s


In [9]:
if __name__ == "__main__":
    np.random.seed(43)
    torch.manual_seed(43)
    name_ls = ["gw", "energy", "tlb", "sgw_numint", "functional_sgw"]
    Ms, heights, features, labels = load_FAUST(1000)
    mats, times = init_matrices(len(Ms))
    st = time.time()
    compute_pairwise(Ms, heights, features, mats, times, alpha=0.0, name_ls=name_ls, grid_n=10, n_projections=100)
    print(f"Pairwise computation finished in {time.time() - st:.2f}s")
    evaluate_all(mats, labels, times, name_ls, k=3, test_size=0.75, trials=1000)

Loading and processing FAUST


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [2:47:38<00:00, 100.58s/it]


Pairwise computation finished in 10058.48s
gw           | kNN acc: 0.330 ± 0.053 | Avg time: 1.8784s
energy       | kNN acc: 0.418 ± 0.059 | Avg time: 0.0307s
tlb          | kNN acc: 0.402 ± 0.060 | Avg time: 0.0863s
sgw_numint   | kNN acc: 0.394 ± 0.056 | Avg time: 0.0178s
functional_sgw | kNN acc: 0.427 ± 0.059 | Avg time: 0.0179s
